# camelsch — Interactive Demo

This notebook demonstrates the **camelsch** Python API using the small synthetic
fixture dataset shipped in `tests/fixtures/`. No download is required.

The fixture data mimics the real CAMELS-CH layout with 3 basins and 5 daily time
steps (1981-01-01 to 1981-01-05).

In [ ]:
from pathlib import Path

DATA_DIR = Path("tests/fixtures/camels_ch")
print(f"Using fixture data at: {DATA_DIR}")

**Using your own data?** After running `camelsch download`, just change `DATA_DIR`
to point at the downloaded dataset — e.g. `DATA_DIR = Path("./data/CAMELS_CH")` or
wherever you passed `--dest`. The rest of this notebook works the same way.

## 1. List Basins and Variables

In [ ]:
import camelsch

basins = camelsch.list_basins(DATA_DIR)
print(f"Basins ({len(basins)}): {basins}")

variables = camelsch.list_variables(DATA_DIR)
print(f"\nVariables ({len(variables)}):")
for v in variables:
    print(f"  - {v}")

## 2. Load Static Attributes

In [ ]:
attrs = camelsch.load_attributes(DATA_DIR)
attrs

In [ ]:
# Filter to specific basins
subset = camelsch.load_attributes(DATA_DIR, basin_ids=["2004", "2007"])
subset[["gauge_name", "area", "elev_mean"]]

## 3. Load Time Series

In [ ]:
# Load a single basin — obs + sim columns are merged automatically
ts = camelsch.load_basin_timeseries(DATA_DIR, "2004")
print(f"Shape: {ts.shape}")
ts.head()

In [ ]:
# Load multiple basins with variable and date filtering
data = camelsch.load_timeseries(
    DATA_DIR,
    basin_ids=["2004", "2007"],
    variables=["precipitation", "discharge_spec"],
    start_date="1981-01-02",
    end_date="1981-01-04",
)
for bid, df in data.items():
    print(f"\nBasin {bid}:")
    print(df)

## 4. Visualization

In [ ]:
import matplotlib.pyplot as plt

ts = camelsch.load_basin_timeseries(DATA_DIR, "2004")

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

axes[0].bar(ts.index, ts["precipitation"], color="steelblue", label="Precipitation")
axes[0].set_ylabel("mm/d")
axes[0].legend()

axes[1].plot(ts.index, ts["discharge_spec"], color="darkorange", marker="o", label="Obs")
axes[1].plot(ts.index, ts["discharge_spec_sim"], color="grey", linestyle="--", marker="s", label="Sim")
axes[1].set_ylabel("mm/d")
axes[1].legend()

fig.suptitle("Basin 2004 — Precipitation & Discharge")
plt.tight_layout()
plt.show()

## 5. Export

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / "basin_2004.csv"
    camelsch.export_timeseries({"2004": ts}, out, fmt="csv")
    print(f"Exported to {out} ({out.stat().st_size} bytes)")
    # Read back the first few lines
    print(out.read_text()[:300])

## 6. CLI Demo

camelsch also provides a full CLI. Here are a few examples using the fixture data:

In [ ]:
!camelsch --version

In [ ]:
!camelsch basins --data-dir tests/fixtures/camels_ch --format json

In [ ]:
!camelsch info --data-dir tests/fixtures/camels_ch